In [1]:
# ROI Plotting for Future/Past Analysis

In [2]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import sys
sys.path.append('/scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/scripts')
import tfsplt_future_past_utils as pu
import warnings
warnings.filterwarnings('ignore')

## Configuration

In [ ]:
# Configuration
res_d = "/scratch/gpfs/HASSON/ij9216/projects/code/247/247-encoding-dev/results/tfs"
output_dir = '/scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/results/roi_figures/future_past/rois_pca'
thresh_joint = 0.12

# lags in data
lag_range_ms = 60000  # The full range in the data
# Define lags (plot)
lag_start = -5000
lag_end = 5000
lag_step = 50
lags_to_plot = np.arange(lag_start, lag_end + lag_step, lag_step).tolist()

# Define colormaps for each label3 (hex colors)
colormaps = {
    'sentence': '#0D8D00',   # Future - medium green
    'sentence2': '#A90008',  # Past - medium red
    'word': '#AC7000',       # Word - orange
    'joint': '#092C72'       # Joint - medium blue
    # 'sentence': '#0a8f00',   # Future - medium green
    # 'sentence2': '#c9001a',  # Past - medium red
    # 'word': '#d67600',       # Word - orange
    # 'joint': '#1a5fb4'       # Joint - medium blue
    # 'sentence': '#085300',   # Future - dark green
    # 'sentence2': '#630005',  # Past - dark red
    # 'word': '#654200',       # Word - brown/orange
    # 'joint': '#061943'   
}

print(f"Results directory: {res_d}")
print(f"Output directory: {output_dir}")
print(f"Joint threshold: {thresh_joint}")
print(f"Lag range: {lag_start} to {lag_end} ms")
print(f"Number of lags: {len(lags_to_plot)}")
print(f"\nColormaps:")
for key, color in colormaps.items():
    print(f"  {key}: {color}")

Results directory: /scratch/gpfs/HASSON/ij9216/projects/code/247/247-encoding-dev/results/tfs
Output directory: /scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/results/roi_figures/future_past/rois_pca
Joint threshold: 0.12
Lag range: -5000 to 5000 ms
Number of lags: 201

Colormaps:
  sentence: #0D8D00
  sentence2: #A90008
  word: #AC7000
  joint: #092C72


## Load Data

In [11]:
# Load comprehension data
print("Loading comprehension data...")
comp_data = pu.load_res_add_roi_threshold(
    f"{res_d}/ij-tfs-%s-gpt2-xl-bandedRidge-lag60-50-all-static_future_past-reph-translate-control_pca300_drop-short_mistral_comp.csv",
    thresh_joint
)

Loading comprehension data...


In [12]:
# Load production data
print("Loading production data...")
prod_data = pu.load_res_add_roi_threshold(
    f"{res_d}/ij-tfs-%s-gpt2-xl-bandedRidge-lag60-50-all-static_future_past-reph-translate-control_pca300_drop-short_mistral_prod.csv",
    thresh_joint
)

Loading production data...


## Drop Problematic Electrodes

In [13]:
# Drop specified electrodes from comprehension (798 G1, 798 G65)
comp_before = len(comp_data)
comp_data = comp_data[~((comp_data['subject'] == '798') & 
                        (comp_data['electrode'].isin(['G1', 'G65'])))]
comp_after = len(comp_data)
print(f"Comprehension: Dropped {comp_before - comp_after} rows (798 G1, G65)")

# Drop specified electrodes from production (798 G104)
prod_before = len(prod_data)
prod_data = prod_data[~((prod_data['subject'] == '798') & 
                        (prod_data['electrode'] == 'G104'))]
prod_after = len(prod_data)
print(f"Production: Dropped {prod_before - prod_after} rows (798 G104)")

print(f"\nFinal shapes:")
print(f"  Comprehension: {comp_data.shape}")
print(f"  Production: {prod_data.shape}")

Comprehension: Dropped 0 rows (798 G1, G65)
Production: Dropped 0 rows (798 G104)

Final shapes:
  Comprehension: (732, 2406)
  Production: (732, 2406)


## Setup Plotting Arguments

In [ ]:
# Get all numeric columns (they're stored as strings '0', '1', '2', etc.)
# Convert them to integers for proper indexing
numeric_cols = [col for col in comp_data.columns if col.isdigit()]
numeric_col_indices = sorted([int(col) for col in numeric_cols])

# The data was encoded with 60 second range at 50ms steps
# That's -60000 to +60000 ms in 50ms steps = 2401 lags
# Column index 0 = lag -60000ms, column index 1200 = lag 0ms, column index 2400 = lag 60000ms
# lag_range_ms = 60000  # The full range in the data
n_lags = len(numeric_col_indices)
lag_values = np.linspace(-lag_range_ms, lag_range_ms, n_lags).astype(int)

# Create mapping from lag values to column names
lag_to_col = {lag_val: str(idx) for idx, lag_val in enumerate(lag_values)}

print(f"Data contains {n_lags} lag columns")
print(f"Lag range: {lag_values[0]} to {lag_values[-1]} ms")
print(f"Lag step: {lag_values[1] - lag_values[0]} ms")

# Get ROIs
rois = sorted([str(roi) for roi in comp_data['roi'].unique() if roi is not None])
print(f"Number of ROIs: {len(rois)}")
print(f"ROIs: {rois}")

# Setup arguments for plot_roi function
class Args:
    def __init__(self, lags, rois, res_dir, colors, lines, legends):
        self.lags = lags
        self.rois = rois
        self.res_dir = res_dir
        self.colors = colors
        self.lines = lines
        self.legends = legends

args = Args(
    lags={
        'lags_all': lag_values,  # All available lag values in milliseconds
        'lags_plt': lags_to_plot,  # Lags to plot (-5 to 5 seconds)
        'lag_ticks': np.arange(-5000, 5001, 1000).tolist(),  # Tick positions
        'lag_tick_labels': np.arange(-5, 6, 1).tolist()  # Tick labels in seconds
    },
    rois=rois,
    res_dir=output_dir,
    colors=[colormaps['sentence'], colormaps['sentence2'], colormaps['word'], colormaps['joint']],
    lines=['sentence', 'sentence2', 'word', 'joint'],
    legends=['Future', 'Past', 'Word', 'Joint']
)

print("\nPlotting configuration:")
print(f"  Lines: {args.lines}")
print(f"  Colors: {args.colors}")
print(f"  Legends: {args.legends}")

Data contains 2401 lag columns
Lag range: -60000 to 60000 ms
Lag step: 50 ms
Number of ROIs: 29
ROIs: ['IFG', 'ITG', 'Left-Amygdala', 'Left-Cerebral-White-Matter', 'MTG', 'Right-Cerebral-White-Matter', 'STG', 'TP', 'Unknown', 'bankssts', 'caudalmiddlefrontal', 'ctx-lh-caudalanteriorcingulate', 'ctx-lh-insula', 'ctx-lh-parstriangularis', 'ctx-lh-precentral', 'ctx-lh-superiortemporal', 'ctx-rh-middletemporal', 'ctx-rh-transversetemporal', 'entorhinal', 'fusiform', 'inferiorparietal', 'lateralorbitofrontal', 'lingual', 'nan', 'postCG', 'preCG', 'rostralmiddlefrontal', 'superiorfrontal', 'supramarginal']

Plotting configuration:
  Lines: ['sentence', 'sentence2', 'word', 'joint']
  Colors: ['#0D8D00', '#A90008', '#AC7000', '#092C72']
  Legends: ['Future', 'Past', 'Word', 'Joint']


## Plot Comprehension ROIs

In [15]:
print("=" * 80)
print("PLOTTING COMPREHENSION ROIs")
print("=" * 80)

# Prepare data in dictionary format as expected by plot_roi
comp_dict = {}
for line in args.lines:
    for roi in args.rois:
        # Filter data for this specific line (label3) and ROI
        subset = comp_data[(comp_data['label3'] == line) & (comp_data['roi'] == roi)]
        if len(subset) > 5:
            comp_dict[(line, roi)] = subset
        #     print(f"  {line} + {roi}: {len(subset)} electrodes")
        # else:
        #     print(f"  {line} + {roi}: NO DATA (skipping)")

# Plot comprehension ROIs
pu.plot_roi(args, comp_dict, mode="comp", ymax=0.2, save='svg', plot_indiv=False)
pu.plot_roi_sep_context(args, comp_dict, mode="comp", ymax=0.2, save='svg', context_thresh_f=0, context_thresh_p=0)
# pu.plot_roi_sep_context(args, comp_dict, mode="comp", ymax=0.2, save='svg', context_thresh_f=350, context_thresh_p=350)
print("\nComprehension ROI plots saved!")

PLOTTING COMPREHENSION ROIs

Comprehension ROI plots saved!


## Plot Production ROIs

In [16]:
print("=" * 80)
print("PLOTTING PRODUCTION ROIs")
print("=" * 80)

# Prepare data in dictionary format as expected by plot_roi
prod_dict = {}
for line in args.lines:
    for roi in args.rois:
        # Filter data for this specific line (label3) and ROI
        subset = prod_data[(prod_data['label3'] == line) & (prod_data['roi'] == roi)]
        if len(subset) > 5:
            prod_dict[(line, roi)] = subset
        #     print(f"  {line} + {roi}: {len(subset)} electrodes")
        # else:
        #     print(f"  {line} + {roi}: NO DATA (skipping)")

# Plot production ROIs
pu.plot_roi(args, prod_dict, mode="prod", ymax=0.2, save='svg', plot_indiv=False)
pu.plot_roi_sep_context(args, prod_dict, mode="prod", ymax=0.2, save='svg', context_thresh_f=0, context_thresh_p=0)

print("\nProduction ROI plots saved!")

PLOTTING PRODUCTION ROIs

Production ROI plots saved!
